In [ ]:
import json
import re
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from scipy.stats import pearsonr
import nltk
from nltk.corpus import stopwords
#nltk.download('stopwords')

In [ ]:
transcript_direction = Path('/Users/dylan/Desktop/Research-Project/Earning-calls-sentiment/data/Raw/transcripts')
def load_transcript_text(filepath):
    with open(filepath) as f:
        data = json.load(f)
    if data and isinstance(data, list) and len(data) > 0:
        return data[0].get('content', '')
    return ''

manifest = []
for filepath in transcript_direction.rglob('*.json'):
    symbol = filepath.parent.name
    stem = filepath.stem
    match = re.match(r'(\d{4})Q(\d)', stem)
    if match:
        year, quarter = int(match.group(1)), int(match.group(2))
        manifest.append({
            'symbol': symbol,
            'year': year,
            'quarter': quarter,
            'filepath': filepath,
        })
manifest_df = pd.DataFrame(manifest)

sample = manifest_df[
    manifest_df['symbol'].isin(['AAPL', 'MSFT', 'JPM', 'KO', 'TSLA'])
].head(100)

In [ ]:
sample = sample.copy()
sample['content'] = sample['filepath'].apply(load_transcript_text)
print(f"Loaded {len(sample)} transcripts")
print(sample[['symbol', 'year', 'quarter']].head())

In [ ]:
np.random.seed(42)
sample['label'] = np.random.normal(0, 0.05, size=len(sample))

In [ ]:
stop_words = set(stopwords.words('english'))

stop_words.update(['company', 'quarter', 'year', 'revenue', 'earnings', 
                   'business', 'results', 'questions', 'call', 'thank',
                   'good', 'morning', 'afternoon', 'operator'])


vectorizer = CountVectorizer(
    lowercase=True,
    stop_words=list(stop_words),
    min_df=5,
    max_df=0.8,
    token_pattern=r'\b[a-z]{3,}\b',
)

X = vectorizer.fit_transform(sample['content'])
vocab = vectorizer.get_feature_names_out()

print(f"Document-term matrix shape: {X.shape}")
print(f"Number of unique words: {len(vocab)}")
print(f"Sample vocab: {vocab[:30]}")

In [ ]:
X_dense = X.toarray()
labels = sample['label'].values

correlations = []
for j in range(len(vocab)):
    word_counts = X_dense[:, j]
    if word_counts.std() == 0:
        correlations.append(0.0)
        continue
    corr, _ = pearsonr(word_counts, labels)
    correlations.append(corr)

screening_results = pd.DataFrame({
    'word': vocab,
    'correlation': correlations,
    'doc_frequency': (X_dense > 0).sum(axis=0),
})
screening_results = screening_results.sort_values('correlation', ascending=False)

print("Top 20 'positive' words (highest correlation with positive label):")
print(screening_results.head(20))

print("\nTop 20 'negative' words (lowest correlation with negative label):")
print(screening_results.tail(20))

In [ ]:
def score_document_sestm(text, vectorizer, correlations):
    """Score a document using SESTM-screened word weights."""
    counts = vectorizer.transform([text]).toarray().flatten()
    return float((counts * correlations).sum())

test_text = load_transcript_text(manifest_df.iloc[0]['filepath'])
score = score_document_sestm(test_text, vectorizer, np.array(correlations))
print(f"SESTM score: {score:.4f}")